# 🌿 Plant Disease Detection — EfficientNetV2S Training

This notebook trains the production model using the same code as the
`src/` modules in the GitHub repo. It uses:
- **EfficientNetV2S** with ImageNet transfer learning (not a custom CNN)
- **Kaggle API** dataset download (faster than Google Drive)
- **ModelCheckpoint** for automatic resume after disconnections
- **Two-phase training**: head-only → fine-tuning

### How to resume after a Colab disconnect
1. Re-run cells 1–6 (setup + dataset)
2. In Cell 7, set `RESUME = True` and set `INITIAL_EPOCH` to the
   last completed epoch shown in the training output
3. Run from Cell 7 onwards

## Cell 1 — Mount Google Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Plant-Disease-Detection'
import os
os.makedirs(f'{BASE_DIR}/models',  exist_ok=True)
os.makedirs(f'{BASE_DIR}/results', exist_ok=True)
os.makedirs(f'{BASE_DIR}/logs',    exist_ok=True)
print('Drive mounted and folders ready ✅')

## Cell 2 — Keep-alive (prevents Colab from disconnecting)

Run this cell, then immediately run Cell 3 onwards.
This JS snippet clicks the 'Connect' button every 60 seconds.

In [ ]:
# Paste into browser console (F12 → Console tab) for keep-alive
# OR run this cell to display the snippet for easy copy-paste
KEEPALIVE_JS = """
// Paste this into your browser DevTools console (F12)
function clickConnect() {
  var btn = document.querySelector('colab-connect-button');
  if (btn) { btn.shadowRoot.querySelector('#connect').click(); }
}
setInterval(clickConnect, 60000);
console.log('Keep-alive started — Colab will not disconnect ✅');
"""
print('Copy the JS below into your browser Console (F12 → Console):')
print('─' * 60)
print(KEEPALIVE_JS)

## Cell 3 — Install extra dependencies

Do NOT install tensorflow here — Colab pre-installs the correct GPU version.

In [ ]:
!pip install -q pyyaml scikit-learn seaborn
print('Dependencies installed ✅')

## Cell 4 — Verify GPU

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if not gpus:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → GPU')
else:
    print('GPU ready ✅')

## Cell 5 — Download PlantVillage dataset via Kaggle API

This is **much faster** than unzipping from Google Drive.
Make sure you have uploaded your `kaggle.json` API key to Drive.

In [ ]:
import os

DATASET_DIR = '/content/PlantVillage'

if os.path.exists(DATASET_DIR) and len(os.listdir(DATASET_DIR)) > 0:
    print(f'Dataset already present at {DATASET_DIR} ✅')
else:
    print('Downloading dataset via Kaggle API...')

    # Copy kaggle.json from Drive to the correct location
    os.makedirs('/root/.config/kaggle', exist_ok=True)
    !cp '/content/drive/MyDrive/kaggle.json' /root/.config/kaggle/kaggle.json
    !chmod 600 /root/.config/kaggle/kaggle.json

    # Download and unzip to /content (fast local SSD)
    !pip install -q kaggle
    !kaggle datasets download -d emmarex/plantdisease -p /content/ --unzip

    # The dataset unzips into /content/PlantVillage
    if not os.path.exists(DATASET_DIR):
        # Some Kaggle versions extract to a subfolder
        extracted = [d for d in os.listdir('/content') if 'plant' in d.lower() or 'village' in d.lower()]
        if extracted:
            os.rename(f'/content/{extracted[0]}', DATASET_DIR)

    print('Download complete ✅')

classes = sorted(os.listdir(DATASET_DIR))
print(f'Classes found: {len(classes)}')
print(classes)

## Cell 6 — Clone repo and set up src imports

In [ ]:
import sys

REPO_DIR = '/content/drive/MyDrive/Plant-Disease-Detection/repo'
os.makedirs(REPO_DIR, exist_ok=True)

# If you have the repo cloned to Drive, add it to the path
# Otherwise, we copy the src files manually below
if os.path.exists(f'{REPO_DIR}/src'):
    sys.path.insert(0, REPO_DIR)
    print(f'Using repo at {REPO_DIR} ✅')
else:
    # Fallback: define helpers inline (mirrors src/ modules exactly)
    print('Repo not found on Drive — using inline module definitions')
    print('(Clone your GitHub repo to Drive for the cleanest setup)')

## Cell 7 — Configuration

**To resume after a disconnect:**
- Set `RESUME = True`
- Set `INITIAL_EPOCH_HEAD` to the last Phase 1 epoch completed
- Set `INITIAL_EPOCH_FINETUNE` to the last Phase 2 epoch completed (0 if Phase 2 hasn't started)

In [ ]:
# ── Training configuration ────────────────────────────────────────────────
DATA_DIR   = DATASET_DIR   # /content/PlantVillage (fast local SSD)
SAVE_DIR   = f'{BASE_DIR}/models'
RESULTS_DIR = f'{BASE_DIR}/results'
IMG_SIZE   = 224
BATCH_SIZE = 32
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.10
SEED       = 42

# Phase 1 — head only
EPOCHS_HEAD      = 10
LR_HEAD          = 1e-3

# Phase 2 — fine-tuning
EPOCHS_FINETUNE  = 20
LR_FINETUNE      = 1e-5
UNFREEZE_LAYERS  = 40

DROPOUT_RATE     = 0.30
DENSE_UNITS      = 256

# ── Resume settings (change these after a disconnect) ─────────────────────
RESUME                  = False  # Set True to resume
INITIAL_EPOCH_HEAD      = 0      # Last completed Phase 1 epoch
INITIAL_EPOCH_FINETUNE  = 0      # Last completed Phase 2 epoch (0 = not started)

print('Configuration loaded ✅')
print(f'  Data:        {DATA_DIR}')
print(f'  Save dir:    {SAVE_DIR}')
print(f'  Resume:      {RESUME}')
if RESUME:
    print(f'  Phase 1 resume epoch: {INITIAL_EPOCH_HEAD}')
    print(f'  Phase 2 resume epoch: {INITIAL_EPOCH_FINETUNE}')

## Cell 8 — Build tf.data datasets

In [ ]:
import random
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split

# ── Reproducibility ──────────────────────────────────────────────────────
import os
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Collect image paths and labels ───────────────────────────────────────
data_dir    = Path(DATA_DIR)
class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
class_to_idx = {c: i for i, c in enumerate(class_names)}
VALID_EXTS   = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

all_paths, all_labels = [], []
for cls in class_names:
    for p in (data_dir / cls).iterdir():
        if p.suffix in VALID_EXTS:
            all_paths.append(str(p))
            all_labels.append(class_to_idx[cls])

print(f'Total images: {len(all_paths)} across {len(class_names)} classes')

# ── Stratified 3-way split ────────────────────────────────────────────────
X_tv, X_test, y_tv, y_test = train_test_split(
    all_paths, all_labels, test_size=TEST_SPLIT,
    stratify=all_labels, random_state=SEED
)
adj_val = VAL_SPLIT / (1.0 - TEST_SPLIT)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=adj_val,
    stratify=y_tv, random_state=SEED
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

# ── Dataset builders ─────────────────────────────────────────────────────
def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    # IMPORTANT: use EfficientNetV2S preprocess_input, NOT /255
    img = tf.keras.applications.efficientnet_v2.preprocess_input(img)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    img = tf.image.random_saturation(img, lower=0.8, upper=1.2)
    return img, label

def make_ds(paths, labels, do_augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if do_augment:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=256, seed=SEED)
    if not do_augment:
        ds = ds.cache()  # cache val/test — they never change
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_train, y_train, do_augment=True,  shuffle=True)
val_ds   = make_ds(X_val,   y_val,   do_augment=False, shuffle=False)
test_ds  = make_ds(X_test,  y_test,  do_augment=False, shuffle=False)

print('Datasets ready ✅')

## Cell 9 — Build EfficientNetV2S model

In [ ]:
def build_model(num_classes, img_size=224, dropout_rate=0.3, dense_units=256):
    base = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3)
    )
    base.trainable = False  # Phase 1: freeze base

    inputs = tf.keras.Input(shape=(img_size, img_size, 3))
    x = base(inputs, training=False)  # keeps BN in inference mode
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Dense(dense_units, activation='relu')(x)
    x = tf.keras.layers.Dropout(dropout_rate / 2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs, name='plant_disease_classifier')


CHECKPOINT_PATH = f'{SAVE_DIR}/best_checkpoint.keras'

if RESUME and os.path.exists(CHECKPOINT_PATH):
    print(f'Resuming from checkpoint: {CHECKPOINT_PATH}')
    model = tf.keras.models.load_model(CHECKPOINT_PATH)
else:
    model = build_model(
        num_classes  = len(class_names),
        img_size     = IMG_SIZE,
        dropout_rate = DROPOUT_RATE,
        dense_units  = DENSE_UNITS,
    )

model.summary()
print(f'\nTotal params: {model.count_params():,}')

## Cell 10 — Compute class weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes_arr = np.unique(y_train)
weights     = compute_class_weight('balanced', classes=classes_arr, y=y_train)
class_weight_dict = dict(zip(classes_arr.tolist(), weights.tolist()))

print('Class weights computed:')
for i, name in enumerate(class_names):
    print(f'  {i:2d}  {name:<50s}  {class_weight_dict[i]:.4f}')

## Cell 11 — Callbacks

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    # Saves to Drive — survives Colab disconnections
    tf.keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
]
print(f'Callbacks ready. Checkpoint will save to:\n  {CHECKPOINT_PATH}')

## Cell 12 — Phase 1: Train classification head (base frozen)

In [ ]:
if RESUME and INITIAL_EPOCH_FINETUNE > 0:
    print('Skipping Phase 1 — resuming directly into Phase 2')
    history1 = None
else:
    print('=== Phase 1: Training classification head ===')
    print(f'Epochs: {INITIAL_EPOCH_HEAD} → {EPOCHS_HEAD}')

    model.compile(
        optimizer = tf.keras.optimizers.Adam(learning_rate=LR_HEAD),
        loss      = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics   = ['accuracy']
    )

    history1 = model.fit(
        train_ds,
        validation_data = val_ds,
        epochs          = EPOCHS_HEAD,
        initial_epoch   = INITIAL_EPOCH_HEAD,
        callbacks       = callbacks,
        class_weight    = class_weight_dict,
        verbose         = 1
    )
    print('Phase 1 complete ✅')

## Cell 13 — Phase 2: Fine-tune top layers of EfficientNetV2S

In [ ]:
print('=== Phase 2: Fine-tuning top layers ===')

base_model = model.layers[1]  # EfficientNetV2S is at index 1
base_model.trainable = True

# Freeze all except the last UNFREEZE_LAYERS layers
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Fine-tuning {trainable_count} / {len(base_model.layers)} base layers')
print(f'Epochs: {INITIAL_EPOCH_FINETUNE} → {EPOCHS_FINETUNE}')

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=LR_FINETUNE),
    loss      = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics   = ['accuracy']
)

history2 = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = EPOCHS_FINETUNE,
    initial_epoch   = INITIAL_EPOCH_FINETUNE,
    callbacks       = callbacks,
    class_weight    = class_weight_dict,
    verbose         = 1
)
print('Phase 2 complete ✅')

## Cell 14 — Save final model and metadata to Drive

In [ ]:
import json

FINAL_MODEL_PATH = f'{SAVE_DIR}/final_model'
model.save(FINAL_MODEL_PATH)
print(f'Final model saved to: {FINAL_MODEL_PATH} ✅')

# Save metadata.json (required by streamlit_app.py)
metadata = {
    'class_names':  class_names,
    'num_classes':  len(class_names),
    'img_size':     IMG_SIZE,
    'architecture': 'EfficientNetV2S',
}
METADATA_PATH = f'{BASE_DIR}/models/metadata.json'
with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'Metadata saved to: {METADATA_PATH} ✅')
print(json.dumps(metadata, indent=2))

## Cell 15 — Plot training history

In [ ]:
import matplotlib.pyplot as plt

def combine_history(h1, h2, key):
    vals = h1.history[key] if h1 else []
    if h2:
        vals = vals + h2.history[key]
    return vals

acc      = combine_history(history1, history2, 'accuracy')
val_acc  = combine_history(history1, history2, 'val_accuracy')
loss     = combine_history(history1, history2, 'loss')
val_loss = combine_history(history1, history2, 'val_loss')
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc,     'g-',  label='Train')
ax1.plot(epochs, val_acc, 'b--', label='Validation')
if history1 and history2:
    ax1.axvline(len(history1.history['accuracy']), color='gray',
                linestyle=':', label='Fine-tune start')
ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, loss,     'r-',  label='Train')
ax2.plot(epochs, val_loss, 'b--', label='Validation')
if history1 and history2:
    ax2.axvline(len(history1.history['loss']), color='gray',
                linestyle=':', label='Fine-tune start')
ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plot_path = f'{RESULTS_DIR}/training_history.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved to {plot_path} ✅')

## Cell 16 — Full evaluation on test set

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# ── Collect predictions ───────────────────────────────────────────────────
y_true_all, y_pred_probs_all = [], []
for batch_imgs, batch_labels in test_ds:
    probs = model.predict(batch_imgs, verbose=0)
    y_pred_probs_all.extend(probs)
    y_true_all.extend(batch_labels.numpy())

y_true_all       = np.array(y_true_all)
y_pred_probs_all = np.array(y_pred_probs_all)
y_pred_all       = np.argmax(y_pred_probs_all, axis=1)

# ── Classification report ─────────────────────────────────────────────────
report = classification_report(y_true_all, y_pred_all, target_names=class_names, digits=4)
print(report)

report_path = f'{RESULTS_DIR}/classification_report.txt'
with open(report_path, 'w') as f:
    f.write(report)
print(f'Classification report saved ✅')

# ── Summary metrics ───────────────────────────────────────────────────────
accuracy   = float(np.mean(y_true_all == y_pred_all))
macro_f1   = float(f1_score(y_true_all, y_pred_all, average='macro'))
weighted_f1 = float(f1_score(y_true_all, y_pred_all, average='weighted'))
print(f'\nTest Accuracy:    {accuracy:.4f}')
print(f'Macro F1:         {macro_f1:.4f}')
print(f'Weighted F1:      {weighted_f1:.4f}')

## Cell 17 — Normalised confusion matrix

In [ ]:
cm = confusion_matrix(y_true_all, y_pred_all, normalize='true')

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='.2f', cmap='Greens',
    xticklabels=class_names, yticklabels=class_names, ax=ax
)
ax.set_title('Normalised Confusion Matrix', fontsize=14)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()

cm_path = f'{RESULTS_DIR}/confusion_matrix.png'
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Confusion matrix saved to {cm_path} ✅')

## Cell 18 — Per-class F1 bar chart

In [ ]:
f1_per_class = f1_score(y_true_all, y_pred_all, average=None)

bar_colors = ['#4CAF50' if s >= 0.9 else '#FF9800' if s >= 0.75 else '#F44336'
              for s in f1_per_class]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(class_names, f1_per_class, color=bar_colors)
ax.axhline(0.90, color='red',    linestyle='--', linewidth=1.2, label='0.90 target')
ax.axhline(0.75, color='orange', linestyle='--', linewidth=1.0, label='0.75 baseline')
ax.set_ylim(0, 1.05)
ax.set_title('Per-Class F1 Score')
ax.set_ylabel('F1 Score')
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()

f1_path = f'{RESULTS_DIR}/f1_scores.png'
plt.savefig(f1_path, dpi=150)
plt.show()
print(f'F1 chart saved to {f1_path} ✅')

## Cell 19 — Export to TFLite FP16 (for mobile deployment)

In [ ]:
TFLITE_PATH = f'{SAVE_DIR}/model_fp16.tflite'

converter = tf.lite.TFLiteConverter.from_saved_model(FINAL_MODEL_PATH)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1_048_576
print(f'TFLite model saved: {size_mb:.1f} MB → {TFLITE_PATH} ✅')

## Cell 20 — Quick single-image prediction test

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Change this path to any test image
TEST_IMAGE_PATH = '/content/drive/MyDrive/Plant-Disease-Detection/test_images/test_img.jpeg'

if not os.path.exists(TEST_IMAGE_PATH):
    print('Test image not found — skipping prediction demo')
else:
    img = Image.open(TEST_IMAGE_PATH).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)
    # IMPORTANT: same preprocessing as training
    arr = tf.keras.applications.efficientnet_v2.preprocess_input(arr)

    preds          = model.predict(arr, verbose=0)[0]
    predicted_idx  = int(np.argmax(preds))
    confidence     = float(preds[predicted_idx]) * 100
    predicted_class = class_names[predicted_idx]

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.title(f'Predicted: {predicted_class}\nConfidence: {confidence:.1f}%')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/prediction_demo.png', dpi=150)
    plt.show()

    print(f'\nPrediction: {predicted_class}')
    print(f'Confidence: {confidence:.1f}%')
    print('\nTop 3:')
    for i in np.argsort(preds)[::-1][:3]:
        print(f'  {class_names[i]:<50s}  {preds[i]:.2%}')